In [ ]:
import json

"""
Importo il file json, dove avrò l'elenco dei
prodotti in magazzino e i venduti
"""

class Product:

    """
    Questa è la classe del prodotto con:
    - nome in stringa
    - quantità in intero
    - prezzo di acquisto in float
    - prezzo di vendita in float
    """

    def __init__(self,
                 name: str,
                 quantity: int,
                 sell_price: float,
                 buy_price: float) -> None:
        self.name = name
        self.quantity = quantity
        self.sell_price = sell_price
        self.buy_price =  buy_price

    def serialize(self):
        return {
            "name": self.name,
            "quantity": self.quantity,
            "sell_price" : self.sell_price,
            "buy_price" : self.buy_price
        }


class Shop:

    """
    Questa è la classe del negozio in cui abbiamo 2 elenchi:
    - stock = magazzino contenente i prodotti registrati
    - orders = elenco dei prodotti venduti
    Nel momento in cui viene fatto un ordine di vendita il prodotto
    viene rimosso dal magazzino e aggiunto alla lista dei venduti
    per poter calcolare i profitti
    """

    def __init__(self) -> None:
        self.stock = []
        self.orders = []

    def load_file(self):

        """
        Apro il file json e per passare dal file al dizionario
        senza usare la stringa uso la funzione 'load'
        """

        with open("data.json", "r") as f:
            data = json.load(f)
            stock = data['stock']
            self.stock = [Product(s["name"], s["quantity"], s["sell_price"], s["buy_price"] ) for s in stock]
            self.orders = data["orders"]

    def close_file(self):

        """
        Funzione per la chiusura del programma al comando "chiudi"
        """

        stock = [s.serialize() for s in self.stock]
        data_new = {'stock': stock,
                    'orders': self.orders}
        with open('data.json', 'w') as f:
            json.dump(data_new, f)

    def list_stock(self):

        """
        Funzione per elencare i prodotti presenti in magazzino al comando "elenca"
        """

        if len(self.stock) == 0:
            print("Nessun prodotto in magazzino.")
            return
        print("PRODOTTO QUANTITA' PREZZO")
        for item in self.stock:
            print(f"{item.name} {item.quantity} €{item.sell_price}")

    def add_stock(self):

        """
        Funzione per l'aggiunta di un prodotto al comando "aggiungi"
        """

        products_available = [s.name for s in self.stock]

        name = input("Nome del prodotto: ")
        quantity = 0
        while quantity <= 0:
            try:
                quantity = int(input("Quantità: "))
            except ValueError:
                print('Inserisci un numero intero positivo.')

        if name in products_available:
            for product in self.stock:
                if product.name == name:
                    product.quantity += quantity
            print("Prodotto presente in magazzino. Prezzo già disponibile.")
            print(f"AGGIUNTO: {quantity} X {name}")
            return

        buy_price = 0.0
        while buy_price <= 0.0:
            try:
                buy_price = round(float(input("Prezzo di acquisto: ")), 2)
            except ValueError:
                print('Inserisci un numero intero o decimale positivo.')

        sell_price = 0.0
        while sell_price <= 0.0:
            try:
                sell_price = round(float(input("Prezzo di vendita: ")), 2)
            except ValueError:
                print('Inserisci un numero intero o decimale positivo.')

        self.stock.append(
            Product(name, quantity, sell_price, buy_price)
            )

        print(f"AGGIUNTO: {quantity} X {name}")


    def _add_order_item(self, products_available):

        """
        Funzione per gestire gli errori relativi all'ordine di un prodotto non presente
        in magazzino o in una quantità superiore a quella disponibile
        """

        name = input("Nome del prodotto: ")
        while name not in products_available:
            print("Prodotto non disponibile.")
            name = input("Nome del prodotto: ")

        quantity = 0
        while quantity <= 0:
            try:
                quantity = int(input("Quantità: "))
                try:
                    if quantity > products_available[name]:
                        raise Exception
                except Exception:
                    quantity = 0
                    print(f'La quantità massima disponibile per {name} è {products_available[name]}.')
            except ValueError:
                print('Inserisci un numero intero.')
        return name, quantity

    def _process_order(self, order):

        """
        Funzione per aggiungere un prodotto venduto all'elenco degli ordini e rimuoverlo
        dall'elenco delle disponibilità in magazzino
        """

        check_out = []

        for product_sold in order:
            product_sold_name = product_sold[0]
            product_sold_qty = product_sold[1]
            for product in self.stock:
                if product_sold_name == product.name:
                    product.quantity -= product_sold_qty
                    check_out_item = {
                        "name": product.name,
                        "quantity": product_sold_qty,
                        "buy_price" : product.buy_price,
                        "sell_price" : product.sell_price
                        }
                    check_out.append(check_out_item)
                if product.quantity == 0:
                    self.stock.remove(product)
        print("VENDITA REGISTRATA")
        total = 0
        for item in check_out:
            print(f'- {item["quantity"]} X {item["name"]}: €{item["sell_price"]}')
            total += item["quantity"] * item["sell_price"]
            self.orders.append(item)
        print(f"Totale: €{round(total, 2)}")



    def make_order(self):

        """
        Funzione per registrare una vendita al comando "vendita"
        """

        if len(self.stock) == 0:
            print("Nessun prodotto in magazzino.")
            return
        products_available = {s.name: s.quantity for s in self.stock}
        order = []
        start = True
        end = False

        while start and not end:
            name, quantity = self._add_order_item(products_available)
            order.append((name, quantity))

            while not end:
                try:
                    cmd = input("Aggiungere un altro prodotto? (si/no): ").lower().strip()
                    if cmd not in ['si', 'no']:
                        raise ValueError
                    if cmd == 'no':
                        end = True
                    else:
                        name, quantity = self._add_order_item(products_available)
                        order.append((name, quantity))

                except ValueError:
                    print('Inserisci solo "si" o "no".')

        self._process_order(order)


    def get_profit(self):

        """
        Funzione per ottenere il profitto lordo e netto al comando "profitti"
        """

        gross = 0
        net = 0
        for ord in self.orders:
            gross += ord['quantity'] * ord['sell_price']
            net += ord['quantity'] * ord['sell_price'] - ord['quantity'] * ord['buy_price']

        print(f"Profitto: lordo=€{round(gross, 2)} netto=€{round(net, 2)}")


def main():

    """
    Costruisco il main per l'avvio del programma con
    la struttura dei comandi e il richiamo alle relative funzioni
    """

    shop = Shop()
    shop.load_file()

    print("Benvenuti nel Negozio Vegano!")

    cmd = ""
    while cmd != "chiudi":
        cmd = input("Inserisci un comando: ")

        if cmd == "chiudi":
            shop.close_file()
            print()
            print("Bye bye")

        elif cmd == "elenca":
            print()
            shop.list_stock()
            print()

        elif cmd == "aggiungi":
            print()
            shop.add_stock()
            print()

        elif cmd == "vendita":
            print()
            shop.make_order()
            print()

        elif cmd == "profitti":
            print()
            shop.get_profit()
            print()

        elif cmd == "aiuto":
            print()
            print("""
I comandi disponibili sono i seguenti:

    • aggiungi: aggiungi un prodotto al magazzino
    • elenca: elenca i prodotti in magazzino
    • vendita: registra una vendita effettuata
    • profitti: mostra i profitti totali
    • aiuto: mostra i possibili comandi
    • chiudi: esci dal programma
""")
            print()

        else:
            print()
            print("""
Comando non valido.

I comandi disponibili sono i seguenti:

    • aggiungi: aggiungi un prodotto al magazzino
    • elenca: elenca i prodotti in magazzino
    • vendita: registra una vendita effettuata
    • profitti: mostra i profitti totali
    • aiuto: mostra i possibili comandi
    • chiudi: esci dal programma
""")
            print()






if __name__ == "__main__":
    main()

Benvenuti nel Negozio Vegano!
Inserisci un comando: elenca

PRODOTTO QUANTITA' PREZZO
pasta 10 €2.0

Inserisci un comando: chiudi

Bye bye


In [ ]:
import json

stock_data = []
orders_data = []

data = {
    "stock": stock_data,
    "orders": orders_data
}

with open("data.json", "w") as json_file:
    json.dump(data, json_file, indent=4)
